<style>
  .rsna-hero {padding: 30px 34px; border-radius: 18px; color: white;
    background: linear-gradient(125deg, #112D4E 0%, #176B87 62%, #28A6A1 100%);
    box-shadow: 0 10px 30px rgba(22,50,79,.16); margin: 8px 0 24px 0;}
  .rsna-kicker {font-size: 12px; font-weight: 800; letter-spacing: .14em;
    text-transform: uppercase; opacity: .78;}
  .rsna-title {font-size: 34px; line-height: 1.08; font-weight: 850; margin: 8px 0 10px;}
  .rsna-subtitle {max-width: 850px; font-size: 16px; line-height: 1.55; opacity: .93;}
  .rsna-pills {margin-top: 18px; display: flex; flex-wrap: wrap; gap: 8px;}
  .rsna-pill {padding: 6px 10px; border: 1px solid rgba(255,255,255,.3);
    border-radius: 999px; background: rgba(255,255,255,.10); font-size: 12px; font-weight: 700;}
  .rsna-tldr {padding: 20px 22px; border: 1px solid #DCE5EA; border-radius: 16px;
    background: #FFFFFF; box-shadow: 0 5px 20px rgba(22,50,79,.07); margin: 0 0 24px;}
  .rsna-tldr-title {font-size: 13px; font-weight: 850; letter-spacing: .08em;
    text-transform: uppercase; color: #176B67; margin-bottom: 13px;}
  .rsna-tldr-grid {display: grid; grid-template-columns: repeat(auto-fit,minmax(180px,1fr)); gap: 14px;}
  .rsna-finding {padding-left: 13px; border-left: 3px solid #28A6A1; line-height: 1.35;}
  .rsna-finding b {display: block; font-size: 18px; color: #16324F; margin-bottom: 3px;}
  .rsna-finding span {font-size: 12px; color: #62707F;}
  .rsna-grid {display: grid; grid-template-columns: repeat(auto-fit,minmax(150px,1fr));
    gap: 12px; margin: 12px 0 22px;}
  .rsna-card {padding: 16px 17px; border: 1px solid #DCE5EA; border-radius: 14px;
    background: linear-gradient(180deg,#FFFFFF 0%,#F7FAFB 100%);}
  .rsna-card .value {font-size: 27px; line-height: 1.1; font-weight: 850; color: #16324F;}
  .rsna-card .label {margin-top: 5px; color: #62707F; font-size: 12px; line-height: 1.3;}
  .rsna-callout {padding: 17px 19px; border-left: 5px solid #28A6A1; border-radius: 10px;
    background: #EFF9F7; color: #16324F; margin: 14px 0 20px; line-height: 1.55;}
  .rsna-decision {padding: 22px 24px; border: 1px solid #BDE5DD; border-radius: 16px;
    background: linear-gradient(135deg,#F2FBF9 0%,#FFFFFF 100%); margin: 12px 0 18px;}
  .rsna-decision h3 {margin: 0 0 8px; color: #176B67; font-size: 21px;}
  .rsna-decision p {margin: 5px 0; line-height: 1.5;}
  .dataframe {font-size: 12px !important; border-collapse: separate !important;
    border-spacing: 0 !important; border: 1px solid #E2E9ED !important; border-radius: 10px; overflow: hidden;}
  .dataframe thead th {background: #16324F !important; color: white !important; border: 0 !important;}
  .dataframe tbody tr:nth-child(even) {background: #F5F8FA;}
  .dataframe td, .dataframe th {padding: 7px 9px !important; border-color: #E8EEF1 !important;}
</style>
<div class="rsna-hero">
  <div class="rsna-kicker">RSNA Knee · diagnostic notebook</div>
  <div class="rsna-title">58 gold labels vs 4 report teachers</div>
  <div class="rsna-subtitle">One tiny expert set has to supervise 4,349 unlabelled knee MRIs.
  I tested the obvious teachers, blend, selector, leakage risk, and protocol metadata.
  The boring choice wins—and the reasons matter.</div>
  <div class="rsna-pills">
    <span class="rsna-pill">CPU only</span><span class="rsna-pill">Internet off</span>
    <span class="rsna-pill">Paired evaluation</span><span class="rsna-pill">Leakage audit</span>
    <span class="rsna-pill">Reproducible outputs</span>
  </div>
</div>
<div class="rsna-tldr">
  <div class="rsna-tldr-title">TL;DR · the entire notebook in 20 seconds</div>
  <div class="rsna-tldr-grid">
    <div class="rsna-finding"><b>0.893 AUC</b><span>Steven v4 leads the paired Gold-57 audit.</span></div>
    <div class="rsna-finding"><b>Blending loses</b><span>Rank blend: 0.892 · cross-fit selector: 0.891.</span></div>
    <div class="rsna-finding"><b>204 studies</b><span>sit inside normalised duplicate-report groups.</span></div>
    <div class="rsna-finding"><b>1 protocol bit</b><span>the two released MRI flags are perfectly identical.</span></div>
  </div>
</div>

## What this notebook answers

**Reader promise.** This notebook compares three independently released teacher families
against the only 58 studies with expert labels, quantifies uncertainty with paired study
bootstraps, cross-fits the tempting classwise selector, and audits the MRI-series contract.
It does not train an image model or claim that teacher AUC predicts the leaderboard.

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import HTML, Markdown, display
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 4)

# A compact visual system keeps the notebook readable on both Kaggle themes.
INK = "#16202A"
MUTED = "#62707F"
NAVY = "#16324F"
BLUE = "#2E86AB"
TEAL = "#28A6A1"
MINT = "#8FD5C8"
GOLD = "#F2B134"
CORAL = "#EF6F6C"
PALE = "#EAF2F5"
GRID = "#DCE5EA"
TEACHER_COLORS = {
    "Pilkwang v2": "#7A6FF0",
    "Steven v2": BLUE,
    "Steven v4 blend": TEAL,
    "Lixin confidence-soft": GOLD,
    "Three-teacher rank blend": CORAL,
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED,
    "axes.titlecolor": INK,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "font.size": 10,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "legend.frameon": False,
})

UID = "StudyInstanceUID"
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 2704


def kpi_cards(items: list[tuple[str, str]]) -> None:
    cards = "".join(
        f'<div class="rsna-card"><div class="value">{value}</div><div class="label">{label}</div></div>'
        for value, label in items
    )
    display(HTML(f'<div class="rsna-grid">{cards}</div>'))


def clean_axes(ax: plt.Axes, grid_axis: str = "x") -> None:
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis=grid_axis, alpha=0.75)
    ax.set_axisbelow(True)


def first_file(candidates: list[Path], label: str) -> Path:
    found = [path for path in candidates if path.is_file()]
    if not found:
        raise FileNotFoundError(f"{label}: none of the declared paths exists")
    return found[0]


cwd = Path.cwd().resolve()
project_candidates = [cwd, cwd.parent]
project_root = next((p for p in project_candidates if (p / "data" / "raw" / "train.csv").is_file()), cwd)

competition_roots = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
    project_root / "data" / "raw",
]
competition_root = next((p for p in competition_roots if (p / "train.csv").is_file()), None)
if competition_root is None:
    raise FileNotFoundError("Competition train.csv was not found")

paths = {
    "train": competition_root / "train.csv",
    "series": competition_root / "train_series.csv",
    "pilkwang": first_file([
        Path("/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv"),
        Path("/kaggle/input/rsna-knee-llm-labels/report_labels_v2.csv"),
        project_root / "data" / "teachers" / "pilkwang" / "report_labels_v2.csv",
    ], "Pilkwang v2"),
    "steven_v2": first_file([
        Path("/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v2.csv"),
        Path("/kaggle/input/rsna-knee-llm-report-labels/llm_labels_v2.csv"),
        project_root / "data" / "teachers" / "steven" / "llm_labels_v2.csv",
    ], "Steven v2"),
    "steven_v4": first_file([
        Path("/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"),
        Path("/kaggle/input/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"),
        project_root / "data" / "teachers" / "steven" / "llm_labels_v4_blend.csv",
    ], "Steven v4 blend"),
    "lixin": first_file([
        Path("/kaggle/input/datasets/lixin73/rsna-knee-llm-report-labels-sol56/report_labels_gpt56sol.csv"),
        Path("/kaggle/input/rsna-knee-llm-report-labels-sol56/report_labels_gpt56sol.csv"),
        project_root / "data" / "teachers" / "lixin" / "report_labels_gpt56sol.csv",
    ], "Lixin confidence labels"),
}

display(pd.DataFrame({
    "input": paths.keys(),
    "file": [path.name for path in paths.values()],
    "source": ["Kaggle input" if str(path).startswith("/kaggle/input/") else "local workspace" for path in paths.values()],
}))

## 1. The supervision bottleneck

The competition has study-level MRI labels for only a small expert subset. Every training
study has a report, so most competitive pipelines convert reports into weak labels. The
expert subset can audit those teachers, but 58 studies are too few to support aggressive
per-class model or blend selection.

In [2]:
train = pd.read_csv(paths["train"], dtype={UID: str})
series = pd.read_csv(paths["series"], dtype={UID: str, "SeriesInstanceUID": str})

missing_schema = [column for column in [UID, "Report", *TARGETS] if column not in train.columns]
if missing_schema:
    raise ValueError(f"train.csv is missing columns: {missing_schema}")

gold = train.dropna(subset=TARGETS, how="all").copy()
if not gold[TARGETS].notna().all().all():
    raise ValueError("The expert subset contains partially missing target rows")

support = pd.DataFrame({
    "positive": gold[TARGETS].sum().astype(int),
    "negative": (len(gold) - gold[TARGETS].sum()).astype(int),
})
support["prevalence"] = support["positive"] / len(gold)

summary = pd.Series({
    "training studies": len(train),
    "expert-labelled studies": len(gold),
    "report-only studies": len(train) - len(gold),
    "MRI series": len(series),
    "studies represented in series table": series[UID].nunique(),
}, name="count")
kpi_cards([
    (f"{len(train):,}", "training studies"),
    (f"{len(gold):,}", "expert-labelled studies"),
    (f"{len(train) - len(gold):,}", "report-only studies"),
    (f"{len(series):,}", "MRI series"),
])

display(support.style.format({"prevalence": "{:.1%}"}).background_gradient(
    subset=["prevalence"], cmap="GnBu", vmin=0, vmax=1
))

support_plot = support.sort_values("positive")
fig, ax = plt.subplots(figsize=(10.5, 6.2))
ax.barh(support_plot.index, support_plot["negative"], color=PALE, label="negative")
ax.barh(support_plot.index, support_plot["positive"], left=support_plot["negative"], color=TEAL, label="positive")
for y, (_, row) in enumerate(support_plot.iterrows()):
    ax.text(59.2, y, f"{row['positive']:.0f} pos  ·  {row['prevalence']:.0%}",
            va="center", ha="left", fontsize=9, color=MUTED)
ax.set_xlim(0, 77)
ax.set_xlabel("Expert-labelled studies")
ax.set_title("Gold-58 class balance is uneven", loc="left", pad=12)
ax.text(0, 1.02, "Positive counts range from 7 to 31; every class has both outcomes.",
        transform=ax.transAxes, color=MUTED, fontsize=10)
ax.legend(ncol=2, loc="lower right")
clean_axes(ax)
plt.tight_layout()
plt.show()

## 2. Duplicate reports and split leakage

A report-derived target is deterministic or nearly deterministic within an exact report.
If duplicate reports cross folds, validation can reward memorising the report template rather
than learning MRI evidence. Report hashes therefore belong in the fold contract.

In [3]:
def normalized_report(text: str) -> str:
    value = unicodedata.normalize("NFKC", str(text)).casefold()
    value = re.sub(r"\s+", " ", value).strip()
    return value


train["report_sha256"] = train["Report"].map(lambda x: hashlib.sha256(str(x).encode("utf-8")).hexdigest())
train["normalized_report_sha256"] = train["Report"].map(
    lambda x: hashlib.sha256(normalized_report(x).encode("utf-8")).hexdigest()
)

exact_sizes = train.groupby("report_sha256").size()
normalized_sizes = train.groupby("normalized_report_sha256").size()
duplicate_audit = pd.Series({
    "unique exact reports": exact_sizes.size,
    "studies in exact-duplicate groups": int(exact_sizes[exact_sizes > 1].sum()),
    "exact duplicate groups": int((exact_sizes > 1).sum()),
    "largest exact group": int(exact_sizes.max()),
    "unique normalized reports": normalized_sizes.size,
    "studies in normalized-duplicate groups": int(normalized_sizes[normalized_sizes > 1].sum()),
    "normalized duplicate groups": int((normalized_sizes > 1).sum()),
    "largest normalized group": int(normalized_sizes.max()),
}, name="value")
kpi_cards([
    (f"{int((exact_sizes > 1).sum())}", "exact duplicate groups"),
    (f"{int(exact_sizes[exact_sizes > 1].sum())}", "studies inside exact duplicates"),
    (f"{int((normalized_sizes > 1).sum())}", "normalised duplicate groups"),
    (f"{int(normalized_sizes.max())}", "studies in the largest group"),
])
display(HTML(
    '<div class="rsna-callout"><b>Fold rule:</b> group by <code>normalized_report_sha256</code>. '
    'That keeps all whitespace/case-equivalent report templates on the same side of validation.</div>'
))
display(duplicate_audit.to_frame())

### Copy-paste guardrail: prove that a split is report-safe

Use this after assigning folds. It fails loudly if one normalised report template appears
in more than one fold—a cheap assertion that prevents a surprisingly expensive mistake.

In [4]:
report_group_map = train[[UID, "normalized_report_sha256"]].copy()


def assert_no_report_group_leakage(
    frame: pd.DataFrame,
    fold_column: str,
    group_column: str = "normalized_report_sha256",
) -> None:
    """Raise when one report group crosses validation folds."""
    folds_per_group = frame.groupby(group_column, dropna=False)[fold_column].nunique()
    leaking_groups = folds_per_group[folds_per_group > 1]
    if len(leaking_groups):
        raise AssertionError(
            f"{len(leaking_groups)} report groups cross folds; largest spans "
            f"{int(leaking_groups.max())} folds"
        )


display(HTML(
    '<div class="rsna-callout"><b>Reusable output:</b> this notebook exports '
    '<code>normalized_report_groups.csv</code>, so an image-training notebook can merge the '
    'group key without reprocessing report text.</div>'
))

## 3. The provided protocol flags collapse two physical properties

`Fluid_Sensitive` and `Fat_Suppression` are clinically different. In the released series
table they are perfectly collinear, so a model cannot recover all protocol distinctions from
these flags alone. Competitive image pipelines need DICOM-header recovery or a deliberately
coarser two-state protocol representation.

In [5]:
flag_table = pd.crosstab(series["Fluid_Sensitive"], series["Fat_Suppression"])
plane_table = series.groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]).size().rename("series")
series_per_study = series.groupby(UID).size()

display(flag_table)
display(plane_table.to_frame())
display(series_per_study.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame("series per study"))

fig, ax = plt.subplots(figsize=(6.3, 4.3))
flag_values = flag_table.to_numpy()
flag_image = ax.imshow(flag_values, cmap="Blues", aspect="auto")
for row in range(flag_values.shape[0]):
    for col in range(flag_values.shape[1]):
        value = int(flag_values[row, col])
        ax.text(col, row, f"{value:,}", ha="center", va="center", fontsize=15,
                fontweight="bold", color="white" if value > flag_values.max() * 0.45 else INK)
ax.set_xticks(range(flag_values.shape[1]), [str(value) for value in flag_table.columns])
ax.set_yticks(range(flag_values.shape[0]), [str(value) for value in flag_table.index])
ax.set_xlabel("Fat suppression flag")
ax.set_ylabel("Fluid-sensitive flag")
ax.set_title("The two protocol flags are identical", loc="left", pad=12)
ax.text(0, 1.04, "Every series falls on the diagonal.", transform=ax.transAxes, color=MUTED)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.show()

if not (series["Fluid_Sensitive"].to_numpy() == series["Fat_Suppression"].to_numpy()).all():
    raise AssertionError("The expected flag collinearity changed in this data version")

## 4. Load and align public report teachers

Four score tables are audited:

- Pilkwang v2 graded report scores.
- Steven v2.
- Steven v4 blend.
- Lixin binary labels softened toward 0.5 by the released per-target confidence.

All comparisons use the same 57 expert studies shared by every table. One expert UID is absent
from the Pilkwang table, so a 58-row score for that source would not be paired fairly.

In [6]:
teacher_frames: dict[str, pd.DataFrame] = {
    "Pilkwang v2": pd.read_csv(paths["pilkwang"], dtype={UID: str})[[UID, *TARGETS]],
    "Steven v2": pd.read_csv(paths["steven_v2"], dtype={UID: str})[[UID, *TARGETS]],
    "Steven v4 blend": pd.read_csv(paths["steven_v4"], dtype={UID: str})[[UID, *TARGETS]],
}

lixin_raw = pd.read_csv(paths["lixin"], dtype={UID: str})
lixin_soft = lixin_raw[[UID]].copy()
for target in TARGETS:
    confidence = lixin_raw[f"{target}__conf"].astype(float)
    binary = lixin_raw[target].astype(float)
    lixin_soft[target] = 0.5 + (2.0 * binary - 1.0) * confidence / 2.0
teacher_frames["Lixin confidence-soft"] = lixin_soft

coverage_rows = []
for name, frame in teacher_frames.items():
    coverage_rows.append({
        "teacher": name,
        "rows": len(frame),
        "unique_uids": frame[UID].nunique(),
        "gold_uids_covered": int(gold[UID].isin(frame[UID]).sum()),
        "all_scores_finite": bool(np.isfinite(frame[TARGETS].to_numpy(float)).all()),
    })
coverage = pd.DataFrame(coverage_rows).set_index("teacher")
display(coverage.style.background_gradient(subset=["gold_uids_covered"], cmap="GnBu"))

common_gold_uids = set(gold[UID])
for frame in teacher_frames.values():
    common_gold_uids.intersection_update(frame[UID])

paired_gold = gold[gold[UID].isin(common_gold_uids)].sort_values(UID).reset_index(drop=True)
dropped_gold = sorted(set(gold[UID]) - common_gold_uids)
display(Markdown(f"**Paired expert cohort:** {len(paired_gold)} studies. Dropped UID count: {len(dropped_gold)}."))

y_gold = paired_gold[TARGETS].to_numpy(float)
teacher_pred: dict[str, np.ndarray] = {}
for name, frame in teacher_frames.items():
    aligned = paired_gold[[UID]].merge(frame, on=UID, how="left", validate="one_to_one")
    teacher_pred[name] = aligned[TARGETS].to_numpy(float)


def class_auc_matrix(predictions: dict[str, np.ndarray]) -> pd.DataFrame:
    values = {
        name: [roc_auc_score(y_gold[:, j], pred[:, j]) for j in range(len(TARGETS))]
        for name, pred in predictions.items()
    }
    return pd.DataFrame(values, index=TARGETS).T


teacher_auc = class_auc_matrix(teacher_pred)
teacher_macro = teacher_auc.mean(axis=1).sort_values(ascending=False).rename("macro AUC")
kpi_cards([
    (f"{teacher_macro.index[0]}", "best teacher on paired Gold-57"),
    (f"{teacher_macro.iloc[0]:.3f}", "best observed macro AUC"),
    (f"{len(paired_gold)}", "expert studies shared by all teachers"),
])
display(teacher_macro.to_frame().style.format("{:.4f}").background_gradient(cmap="GnBu", vmin=.80, vmax=.91))

## 5. Per-target evidence is complementary, but noisy

The heatmap explains why a global teacher blend is tempting. Different sources lead on
different targets. It also shows the danger: selecting twelve winners on the same 57 rows is
a high-variance multiple-selection procedure.

In [7]:
fig, ax = plt.subplots(figsize=(13, 4.8))
image = ax.imshow(teacher_auc.to_numpy(), aspect="auto", cmap="YlGnBu", vmin=0.65, vmax=1.0)
ax.set_xticks(np.arange(len(TARGETS)))
ax.set_xticklabels(TARGETS, rotation=55, ha="right")
ax.set_yticks(np.arange(len(teacher_auc)))
ax.set_yticklabels(teacher_auc.index)
ax.set_title("No teacher wins every target", loc="left", pad=14)
ax.text(0, 1.05, "Paired ROC AUC on the same 57 expert-labelled studies · outlined cells are column winners",
        transform=ax.transAxes, color=MUTED, fontsize=10)
for i in range(teacher_auc.shape[0]):
    for j in range(teacher_auc.shape[1]):
        value = teacher_auc.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,
                fontweight="bold" if value == teacher_auc.iloc[:, j].max() else "normal",
                color="white" if value > 0.86 else INK)
        if value == teacher_auc.iloc[:, j].max():
            ax.add_patch(plt.Rectangle((j - .47, i - .47), .94, .94, fill=False,
                                       edgecolor=GOLD, linewidth=2.2))
for spine in ax.spines.values():
    spine.set_visible(False)
colorbar = fig.colorbar(image, ax=ax, pad=.02, fraction=.025)
colorbar.set_label("ROC AUC")
plt.tight_layout()
plt.show()

best_descriptive = pd.DataFrame({
    "best observed teacher": teacher_auc.idxmax(axis=0),
    "best observed AUC": teacher_auc.max(axis=0),
    "worst observed AUC": teacher_auc.min(axis=0),
})
best_descriptive["observed spread"] = best_descriptive["best observed AUC"] - best_descriptive["worst observed AUC"]
display(best_descriptive.sort_values("observed spread", ascending=False).style.format({
    "best observed AUC": "{:.3f}", "worst observed AUC": "{:.3f}", "observed spread": "{:.3f}"
}).background_gradient(subset=["observed spread"], cmap="YlOrBr"))

## 6. Test the obvious global blend

Because ROC AUC depends only on ordering, teachers are percentile-ranked per target before
averaging. The blend uses Pilkwang v2, Steven v2, and Lixin confidence-soft so that one Steven
release does not enter twice.

In [8]:
blend_members = ["Pilkwang v2", "Steven v2", "Lixin confidence-soft"]
ranked_members = []
for name in blend_members:
    pred = teacher_pred[name]
    ranked = np.column_stack([
        rankdata(pred[:, j], method="average") / len(pred)
        for j in range(len(TARGETS))
    ])
    ranked_members.append(ranked)

teacher_pred_with_blend = dict(teacher_pred)
teacher_pred_with_blend["Three-teacher rank blend"] = np.stack(ranked_members).mean(axis=0)
auc_with_blend = class_auc_matrix(teacher_pred_with_blend)
macro_with_blend = auc_with_blend.mean(axis=1).sort_values(ascending=False).rename("macro AUC")
display(macro_with_blend.to_frame())


def auc_pairwise(y_true: np.ndarray, score: np.ndarray) -> float:
    positive = score[y_true == 1]
    negative = score[y_true == 0]
    return float((positive[:, None] > negative).mean() + 0.5 * (positive[:, None] == negative).mean())


def macro_auc_for_indices(name: str, indices: np.ndarray) -> float:
    pred = teacher_pred_with_blend[name]
    return float(np.mean([
        auc_pairwise(y_gold[indices, j], pred[indices, j])
        for j in range(len(TARGETS))
    ]))


rng = np.random.default_rng(SEED)
bootstrap_names = ["Pilkwang v2", "Steven v2", "Steven v4 blend", "Lixin confidence-soft", "Three-teacher rank blend"]
bootstrap_values = {name: [] for name in bootstrap_names}
for _ in range(5000):
    indices = rng.integers(0, len(y_gold), len(y_gold))
    if any(np.unique(y_gold[indices, j]).size < 2 for j in range(len(TARGETS))):
        continue
    for name in bootstrap_names:
        bootstrap_values[name].append(macro_auc_for_indices(name, indices))

bootstrap_summary = []
for name in bootstrap_names:
    values = np.asarray(bootstrap_values[name])
    bootstrap_summary.append({
        "teacher": name,
        "observed macro AUC": macro_auc_for_indices(name, np.arange(len(y_gold))),
        "bootstrap 2.5%": np.quantile(values, 0.025),
        "bootstrap median": np.quantile(values, 0.5),
        "bootstrap 97.5%": np.quantile(values, 0.975),
    })
bootstrap_summary = pd.DataFrame(bootstrap_summary).set_index("teacher")
display(bootstrap_summary.style.format("{:.4f}").background_gradient(
    subset=["observed macro AUC"], cmap="GnBu", vmin=.82, vmax=.90
))

forest = bootstrap_summary.sort_values("observed macro AUC")
fig, ax = plt.subplots(figsize=(9.5, 4.8))
y_positions = np.arange(len(forest))
observed = forest["observed macro AUC"].to_numpy()
lower = forest["bootstrap 2.5%"].to_numpy()
upper = forest["bootstrap 97.5%"].to_numpy()
colors = [TEACHER_COLORS.get(name, BLUE) for name in forest.index]
for y, name, point, low, high, color in zip(y_positions, forest.index, observed, lower, upper, colors):
    ax.plot([low, high], [y, y], color=color, linewidth=5, alpha=.28, solid_capstyle="round")
    ax.scatter(point, y, s=95, color=color, edgecolor="white", linewidth=1.5, zorder=3)
    ax.text(high + .004, y, f"{point:.3f}", va="center", fontsize=9, color=INK)
ax.set_yticks(y_positions, forest.index)
ax.set_xlim(min(lower) - .01, max(upper) + .045)
ax.set_xlabel("Macro ROC AUC")
ax.set_title("Steven v4 leads, but Gold-57 uncertainty is wide", loc="left", pad=12)
ax.text(0, 1.04, "Dots: observed score · bands: paired study-bootstrap 95% intervals",
        transform=ax.transAxes, color=MUTED)
clean_axes(ax)
plt.tight_layout()
plt.show()

blend_values = np.asarray(bootstrap_values["Three-teacher rank blend"])
delta_rows = []
for comparator in bootstrap_names[:-1]:
    delta = blend_values - np.asarray(bootstrap_values[comparator])
    delta_rows.append({
        "comparison": f"blend minus {comparator}",
        "mean delta": delta.mean(),
        "delta 2.5%": np.quantile(delta, 0.025),
        "delta 97.5%": np.quantile(delta, 0.975),
        "bootstrap P(delta > 0)": (delta > 0).mean(),
    })
delta_table = pd.DataFrame(delta_rows).set_index("comparison")
display(delta_table.style.format({
    "mean delta": "{:+.4f}", "delta 2.5%": "{:+.4f}", "delta 97.5%": "{:+.4f}",
    "bootstrap P(delta > 0)": "{:.1%}",
}).background_gradient(subset=["bootstrap P(delta > 0)"], cmap="GnBu", vmin=0, vmax=1))

## 7. Cross-fit the classwise selector

The descriptive per-target winners are selected on the same rows used to score them. To test
whether that apparent advantage survives, this cell uses repeated three-fold multilabel
stratification. For each outer fold and target, the best teacher is chosen using only the
other two folds, then used on the held-out fold. Teacher scores are rank-normalised without
using labels.

In [9]:
selector_candidates = ["Pilkwang v2", "Steven v2", "Steven v4 blend", "Lixin confidence-soft"]
ranked_candidate_pred = {
    name: np.column_stack([
        rankdata(teacher_pred[name][:, j], method="average") / len(y_gold)
        for j in range(len(TARGETS))
    ])
    for name in selector_candidates
}


def iterative_multilabel_folds(y: np.ndarray, seed: int, n_splits: int = 3) -> np.ndarray:
    local_rng = np.random.default_rng(seed)
    n_samples, n_labels = y.shape
    fold = np.full(n_samples, -1, dtype=int)
    remaining = np.ones(n_samples, dtype=bool)
    desired_size = np.full(n_splits, n_samples / n_splits, dtype=float)
    desired_positive = np.tile(y.sum(axis=0) / n_splits, (n_splits, 1)).astype(float)

    while True:
        remaining_positive = y[remaining].sum(axis=0)
        available_labels = np.flatnonzero(remaining_positive > 0)
        if not len(available_labels):
            break
        rare_label = available_labels[np.argmin(remaining_positive[available_labels])]
        samples = np.flatnonzero(remaining & (y[:, rare_label] == 1))
        local_rng.shuffle(samples)
        for sample_index in samples:
            if not remaining[sample_index]:
                continue
            candidates = np.flatnonzero(np.isclose(
                desired_positive[:, rare_label], desired_positive[:, rare_label].max()
            ))
            best_size = desired_size[candidates].max()
            candidates = candidates[np.isclose(desired_size[candidates], best_size)]
            chosen_fold = int(local_rng.choice(candidates))
            fold[sample_index] = chosen_fold
            remaining[sample_index] = False
            desired_size[chosen_fold] -= 1
            desired_positive[chosen_fold] -= y[sample_index]

    for sample_index in local_rng.permutation(np.flatnonzero(remaining)):
        candidates = np.flatnonzero(np.isclose(desired_size, desired_size.max()))
        chosen_fold = int(local_rng.choice(candidates))
        fold[sample_index] = chosen_fold
        desired_size[chosen_fold] -= 1
    return fold


selector_scores = []
choice_counts = {name: 0 for name in selector_candidates}
for repeat in range(50):
    fold = iterative_multilabel_folds(y_gold.astype(int), SEED + repeat, n_splits=3)
    oof = np.zeros_like(y_gold, dtype=float)
    for fold_id in range(3):
        train_mask = fold != fold_id
        valid_mask = ~train_mask
        for target_index in range(len(TARGETS)):
            candidate_scores = [
                roc_auc_score(y_gold[train_mask, target_index], ranked_candidate_pred[name][train_mask, target_index])
                for name in selector_candidates
            ]
            selected = selector_candidates[int(np.argmax(candidate_scores))]
            choice_counts[selected] += 1
            oof[valid_mask, target_index] = ranked_candidate_pred[selected][valid_mask, target_index]
    selector_scores.append(np.mean([
        roc_auc_score(y_gold[:, j], oof[:, j]) for j in range(len(TARGETS))
    ]))

selector_scores = np.asarray(selector_scores)
selector_result = pd.Series({
    "cross-fitted selector mean": selector_scores.mean(),
    "cross-fitted selector standard deviation": selector_scores.std(),
    "cross-fitted selector minimum": selector_scores.min(),
    "cross-fitted selector maximum": selector_scores.max(),
    "Steven v4 fixed": macro_with_blend["Steven v4 blend"],
    "three-teacher fixed blend": macro_with_blend["Three-teacher rank blend"],
}, name="macro AUC")
display(selector_result.to_frame().style.format("{:.4f}").background_gradient(cmap="GnBu", vmin=.86, vmax=.90))

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 4.4), gridspec_kw={"width_ratios": [1.15, 1]})
ax_left.hist(selector_scores, bins=12, color=MINT, edgecolor="white")
ax_left.axvline(macro_with_blend["Steven v4 blend"], color=TEAL, linewidth=2.5,
                label=f"Steven v4 fixed · {macro_with_blend['Steven v4 blend']:.3f}")
ax_left.axvline(macro_with_blend["Three-teacher rank blend"], color=CORAL, linewidth=2.2,
                linestyle="--", label=f"fixed rank blend · {macro_with_blend['Three-teacher rank blend']:.3f}")
ax_left.set_title("Classwise selection does not add signal", loc="left", pad=12)
ax_left.set_xlabel("Cross-fitted macro AUC")
ax_left.set_ylabel("Repeated splits")
ax_left.legend(fontsize=8)
clean_axes(ax_left)

choice_series = pd.Series(choice_counts).sort_values()
choice_colors = [TEACHER_COLORS.get(name, BLUE) for name in choice_series.index]
ax_right.barh(choice_series.index, choice_series.values, color=choice_colors)
for y, value in enumerate(choice_series.values):
    ax_right.text(value + 2, y, f"{value:,}", va="center", color=MUTED, fontsize=9)
ax_right.set_xlim(0, choice_series.max() * 1.18)
ax_right.set_xlabel("Times selected across folds and targets")
ax_right.set_title("Selector choices are unstable", loc="left", pad=12)
clean_axes(ax_right)
plt.tight_layout()
plt.show()

## 8. How different are the teachers on all 4,407 studies?

Diversity is useful only when it is paired with adequate target fidelity. The independent
teachers disagree most on synovitis and fracture, but the lower-fidelity teacher should not
enter a global blend merely because its ranks are different.

In [10]:
agreement_names = ["Pilkwang v2", "Steven v2", "Lixin confidence-soft"]
agreement_rows = []
for left_index, left_name in enumerate(agreement_names):
    for right_name in agreement_names[left_index + 1:]:
        aligned = teacher_frames[left_name].merge(
            teacher_frames[right_name], on=UID, suffixes=("_left", "_right"), validate="one_to_one"
        )
        correlations = {
            target: spearmanr(aligned[f"{target}_left"], aligned[f"{target}_right"]).statistic
            for target in TARGETS
        }
        agreement_rows.append(pd.Series(correlations, name=f"{left_name} vs {right_name}"))
agreement = pd.DataFrame(agreement_rows)
agreement["mean"] = agreement.mean(axis=1)
display(agreement.style.format("{:.3f}").background_gradient(cmap="GnBu", vmin=.2, vmax=1))

agreement_targets = agreement.drop(columns="mean")
fig, ax = plt.subplots(figsize=(12.5, 3.8))
agreement_image = ax.imshow(agreement_targets.to_numpy(), aspect="auto", cmap="GnBu", vmin=.2, vmax=1)
ax.set_xticks(range(len(agreement_targets.columns)), agreement_targets.columns, rotation=52, ha="right")
ax.set_yticks(range(len(agreement_targets.index)), agreement_targets.index)
for row in range(agreement_targets.shape[0]):
    for col in range(agreement_targets.shape[1]):
        value = agreement_targets.iloc[row, col]
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8,
                color="white" if value > .72 else INK)
ax.set_title("Teacher disagreement is target-specific", loc="left", pad=14)
ax.text(0, 1.08, "Spearman rank correlation across all shared training studies",
        transform=ax.transAxes, color=MUTED)
for spine in ax.spines.values():
    spine.set_visible(False)
colorbar = fig.colorbar(agreement_image, ax=ax, pad=.02, fraction=.025)
colorbar.set_label("rank correlation")
plt.tight_layout()
plt.show()

## 9. Decision card

The audit changes the training plan in four concrete ways.

In [11]:
display(HTML(f"""
<div class="rsna-decision">
  <h3>✓ Promote Steven v4 as the default weak-label table</h3>
  <p>It leads the paired audit at <b>{macro_with_blend['Steven v4 blend']:.3f} macro AUC</b>.
  Neither the fixed rank blend nor the cross-fitted classwise selector improves on it.</p>
</div>
<div class="rsna-grid">
  <div class="rsna-card"><div class="value">01</div><div class="label"><b>Group the folds</b><br>
  Keep every normalised report hash inside a single fold.</div></div>
  <div class="rsna-card"><div class="value">02</div><div class="label"><b>Run only narrow ablations</b><br>
  Test Lixin on lateral meniscus and Pilkwang on fracture.</div></div>
  <div class="rsna-card"><div class="value">03</div><div class="label"><b>Recover DICOM metadata</b><br>
  The released protocol flags collapse to one bit.</div></div>
  <div class="rsna-card"><div class="value">04</div><div class="label"><b>Validate the image model</b><br>
  Teacher AUC is not leaderboard AUC; require grouped OOF and paired seeds.</div></div>
</div>
"""))

In [12]:
output_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else project_root / "artifacts"
output_root.mkdir(parents=True, exist_ok=True)

teacher_auc.to_csv(output_root / "gold57_teacher_auc_by_target.csv")
bootstrap_summary.to_csv(output_root / "gold57_teacher_bootstrap_summary.csv")
delta_table.to_csv(output_root / "gold57_teacher_blend_deltas.csv")
report_group_map.to_csv(output_root / "normalized_report_groups.csv", index=False)

machine_summary = {
    "competition": "rsna-knee-abnormality-detection",
    "audit_date": "2026-10-01",
    "train_studies": int(len(train)),
    "expert_studies": int(len(gold)),
    "paired_expert_studies": int(len(paired_gold)),
    "default_teacher": "Steven v4 blend",
    "default_teacher_macro_auc_gold57": float(macro_with_blend["Steven v4 blend"]),
    "three_teacher_rank_blend_macro_auc_gold57": float(macro_with_blend["Three-teacher rank blend"]),
    "cross_fitted_selector_mean_macro_auc": float(selector_scores.mean()),
    "report_grouping_column": "normalized_report_sha256",
    "protocol_flags_identical": True,
}
(output_root / "gold58_teacher_audit_summary.json").write_text(
    json.dumps(machine_summary, indent=2) + "\n", encoding="utf-8"
)
display(Markdown("Audit tables and the machine-readable summary were written to the notebook output directory."))
display(pd.Series(machine_summary, name="value").to_frame())

## Fork it, test it, or prove it wrong

This notebook is deliberately a **diagnostic building block**, not a finished image model.
A useful fork can now merge `normalized_report_groups.csv`, train one fixed architecture on
Steven v4 labels, and change exactly one thing at a time:

1. substitute Lixin only for lateral meniscus;
2. substitute Pilkwang only for fracture;
3. recover sequence and protocol tags from DICOM headers;
4. report paired, report-grouped OOF deltas across seeds.

If a different teacher or blend wins under that contract, the most informative comparison is
the per-target OOF delta—not a single public-LB number. That is the cleanest way to extend or
challenge this audit.

**Question for the community:** which DICOM tags have most reliably separated fluid-sensitive
from fat-suppressed series in this release? A verified tag recipe would unlock the next useful
version of this notebook.

Need a scored inference reference first? The companion [0.943 public-stack reproduction](https://www.kaggle.com/code/skarin/rsna-knee-reproducing-the-0-943-public-stack) freezes the public ensemble, artifact identities, and execution receipts used as our E0 anchor.

## Sources, licenses, and limits

- Competition data: [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection), subject to the competition data terms and MIRA license.
- Pilkwang teacher table: [RSNA Knee LLM-read report labels](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), CC0-1.0.
- Steven teacher tables: [RSNA Knee LLM Report Labels](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), CC0-1.0.
- Lixin teacher table: [RSNA Knee LLM Report Labels Sol56](https://www.kaggle.com/datasets/lixin73/rsna-knee-llm-report-labels-sol56), CC0-1.0.

Scores in this notebook are locally executed teacher-vs-expert measurements on a 57-study
paired subset. They are not image-model scores, clinical validation, public-LB scores, or
private-LB estimates. Bootstrap intervals quantify patient resampling on this small cohort;
they do not cover dataset shift, annotation-process differences, or teacher-generation error.